# 06. Сравнение моделей сентимента

**Сравниваем три источника тональности:**
- `mxlcw/rubert-tiny2-russian-financial-sentiment` — основная модель
- `blanchefort/rubert-base-cased-sentiment-rusentiment` — robustness check
- `MediaIndex` Медиалогии — проприетарный индекс

**Входные файлы:**
- `sentiment_scores.csv` — построчные scores mxlcw (выход ноутбука 03)
- `sentiment_scores_blanchefort.csv` — построчные scores blanchefort
- `medialogy_export.csv` — исходник с колонкой `media_index`

In [ ]:
# ── ЯЧЕЙКА 1: Установка ──────────────────────────────────────
!pip install matplotlib seaborn scipy -q

In [ ]:
# ── ЯЧЕЙКА 2: Загрузка данных ─────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Построчные scores (одна строка = статья × тикер)
mxl = pd.read_csv('/content/sentiment_scores.csv', encoding='utf-8-sig')
bla = pd.read_csv('/content/sentiment_scores_blanchefort.csv', encoding='utf-8-sig')

# Медиалогия — берём только num + media_index
med = pd.read_csv(
    '/content/medialogy_export.csv',
    sep=';', engine='c', lineterminator='\n', encoding='utf-8-sig',
    on_bad_lines='skip',
    usecols=lambda c: c.strip().lower() in ('num', 'media_index', 'visibility'),
)
med.columns = [c.strip().lower() for c in med.columns]

# Мержим по num (уникальный id статьи)
df = (
    mxl[['num', 'sentiment_score']].rename(columns={'sentiment_score': 'score_mxl'})
    .merge(
        bla[['num', 'sentiment_score']].rename(columns={'sentiment_score': 'score_bla'}),
        on='num', how='inner'
    )
    .merge(med, on='num', how='left')
    .drop_duplicates(subset='num')
)

# MediaIndex — нормализуем знак (делим на visibility, если есть)
if 'visibility' in df.columns:
    df['mi_norm'] = df['media_index'] / df['visibility'].replace(0, np.nan)
else:
    df['mi_norm'] = df['media_index']

# Бинарный знак для sign agreement
df['sign_mxl'] = np.sign(df['score_mxl'])
df['sign_bla'] = np.sign(df['score_bla'])
df['sign_mi']  = np.sign(df['mi_norm'])

print(f'Совмещённых записей: {len(df)}')
print(f"\nmxlcw:       mean={df['score_mxl'].mean():.3f}  std={df['score_mxl'].std():.3f}")
print(f"blanchefort: mean={df['score_bla'].mean():.3f}  std={df['score_bla'].std():.3f}")
print(f"MediaIndex:  mean={df['mi_norm'].mean():.3f}  std={df['mi_norm'].std():.3f}")

In [ ]:
# ── ЯЧЕЙКА 3: Описательная статистика ────────────────────────

stats_table = pd.DataFrame({
    'Модель': ['mxlcw (основная)', 'blanchefort (robustness)', 'MediaIndex'],
    'N':      [df['score_mxl'].notna().sum(),
               df['score_bla'].notna().sum(),
               df['mi_norm'].notna().sum()],
    'Mean':   [df['score_mxl'].mean(), df['score_bla'].mean(), df['mi_norm'].mean()],
    'Std':    [df['score_mxl'].std(),  df['score_bla'].std(),  df['mi_norm'].std()],
    'Min':    [df['score_mxl'].min(),  df['score_bla'].min(),  df['mi_norm'].min()],
    'Max':    [df['score_mxl'].max(),  df['score_bla'].max(),  df['mi_norm'].max()],
    '% positive': [
        (df['sign_mxl'] > 0).mean() * 100,
        (df['sign_bla'] > 0).mean() * 100,
        (df['sign_mi']  > 0).mean() * 100,
    ],
    '% negative': [
        (df['sign_mxl'] < 0).mean() * 100,
        (df['sign_bla'] < 0).mean() * 100,
        (df['sign_mi']  < 0).mean() * 100,
    ],
    '% neutral': [
        (df['sign_mxl'] == 0).mean() * 100,
        (df['sign_bla'] == 0).mean() * 100,
        (df['sign_mi']  == 0).mean() * 100,
    ],
}).set_index('Модель').round(3)

print(stats_table.to_string())

In [ ]:
# ── ЯЧЕЙКА 4: Распределения scores (KDE + гистограмма) ───────

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

plot_data = [
    (df['score_mxl'].dropna(), 'mxlcw\n(основная)', '#2563eb'),
    (df['score_bla'].dropna(), 'blanchefort\n(robustness)', '#dc2626'),
    (df['mi_norm'].dropna(),   'MediaIndex\n(Медиалогия)', '#16a34a'),
]

for ax, (series, label, color) in zip(axes, plot_data):
    ax.hist(series, bins=60, alpha=0.4, color=color, density=True)
    series.plot.kde(ax=ax, color=color, lw=2)
    ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Score')
    ax.text(0.97, 0.95, f'std={series.std():.3f}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, color=color)

fig.suptitle('Распределение сентимент-скоров', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/content/fig_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── ЯЧЕЙКА 5: Корреляционная матрица ─────────────────────────

corr_df = df[['score_mxl', 'score_bla', 'mi_norm']].dropna()
corr_df.columns = ['mxlcw', 'blanchefort', 'MediaIndex']

pearson = corr_df.corr(method='pearson')
spearman = corr_df.corr(method='spearman')

print('Pearson:')
print(pearson.round(3).to_string())
print('\nSpearman:')
print(spearman.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (corr_mat, title) in zip(axes, [(pearson, 'Pearson'), (spearman, 'Spearman')]):
    sns.heatmap(
        corr_mat, annot=True, fmt='.3f', cmap='RdBu_r',
        vmin=-1, vmax=1, center=0,
        square=True, ax=ax, cbar_kws={'shrink': 0.8},
    )
    ax.set_title(f'Корреляция ({title})', fontsize=11)

plt.tight_layout()
plt.savefig('/content/fig_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── ЯЧЕЙКА 6: Sign agreement ──────────────────────────────────

pairs = [
    ('sign_mxl', 'sign_bla', 'mxlcw vs blanchefort'),
    ('sign_mxl', 'sign_mi',  'mxlcw vs MediaIndex'),
    ('sign_bla', 'sign_mi',  'blanchefort vs MediaIndex'),
]

print('Sign agreement (доля совпадений знака, нейтральные исключены):')
for col_a, col_b, label in pairs:
    sub = df[[col_a, col_b]].dropna()
    sub = sub[(sub[col_a] != 0) & (sub[col_b] != 0)]
    agree = (sub[col_a] == sub[col_b]).mean()
    print(f'  {label:<35}  {agree:.3f}  (N={len(sub)})')

In [ ]:
# ── ЯЧЕЙКА 7: Scatter mxlcw vs blanchefort ───────────────────

sub = df[['score_mxl', 'score_bla']].dropna()
r, p = stats.pearsonr(sub['score_mxl'], sub['score_bla'])

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sub['score_mxl'], sub['score_bla'],
           alpha=0.15, s=8, color='#2563eb', rasterized=True)
ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.axvline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('mxlcw score')
ax.set_ylabel('blanchefort score')
ax.set_title(f'mxlcw vs blanchefort  (r={r:.3f}, p={p:.3f})', fontsize=11)
plt.tight_layout()
plt.savefig('/content/fig_scatter_models.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── ЯЧЕЙКА 8: Примеры расхождений ────────────────────────────
# Статьи, где mxlcw и blanchefort сильно расходятся

df['delta'] = (df['score_mxl'] - df['score_bla']).abs()
worst = df.nlargest(10, 'delta')[['num', 'title', 'score_mxl', 'score_bla', 'mi_norm', 'delta']]

print('Топ-10 статей по расхождению mxlcw vs blanchefort:')
pd.set_option('display.max_colwidth', 80)
print(worst.to_string(index=False))

In [ ]:
# ── ЯЧЕЙКА 9: Дневная агрегация — сравнение трендов ──────────

daily_mxl = pd.read_csv('/content/sentiment_daily.csv', encoding='utf-8-sig',
                        parse_dates=['date_day'])
daily_bla = pd.read_csv('/content/sentiment_daily_blanchefort.csv', encoding='utf-8-sig',
                        parse_dates=['date_day'])

# Агрегируем по всем тикерам на дату (рыночный агрегат)
mkt_mxl = daily_mxl.groupby('date_day')['sent_mean'].mean().rename('mxlcw')
mkt_bla = daily_bla.groupby('date_day')['sent_mean'].mean().rename('blanchefort')

daily_compare = pd.concat([mkt_mxl, mkt_bla], axis=1).dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, (col, color) in zip(axes, [('mxlcw', '#2563eb'), ('blanchefort', '#dc2626')]):
    ax.plot(daily_compare.index, daily_compare[col],
            color=color, lw=0.8, alpha=0.7)
    # 30-дневное скользящее среднее
    ax.plot(daily_compare.index,
            daily_compare[col].rolling(30).mean(),
            color=color, lw=2, label='MA30')
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.axvline(pd.Timestamp('2022-02-24'), color='red', lw=1, ls=':', alpha=0.7,
               label='24.02.2022')
    ax.set_ylabel(col)
    ax.legend(fontsize=8)

axes[0].set_title('Дневной агрегированный сентимент (рыночный уровень)', fontsize=12)
plt.tight_layout()
plt.savefig('/content/fig_daily_sentiment.png', bbox_inches='tight', dpi=150)
plt.show()